In [28]:

import sys
import os


sys.path.append(os.path.abspath(".."))


from src.preprocess import pull_pitcher_data
from datetime import date
from dateutil.relativedelta import relativedelta



pitcher_first = 'Chris'
pitcher_last = 'Sale'



end_date = date.today().strftime("%Y-%m-%d")
start_date = (date.today() - relativedelta(years=2)).strftime("%Y-%m-%d")


df = pull_pitcher_data(pitcher_first, pitcher_last,start_date, end_date)
df.columns

Gathering Player Data


Index(['pitch_type_map', 'at_bat_number', 'pitch_type_map', 'balls', 'strikes',
       'outs_when_up', 'home_score_diff', 'runners_on_base', 'inning_top',
       'prev_pitch_1', 'prev_pitch_2', 'prev_pitch_3', 'batter_is_right',
       'pitcher_is_right'],
      dtype='str')

In [12]:

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
# Metrics
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_curve,
    auc,
    confusion_matrix,
    classification_report,
)

# Plotting (for ROC curve and confusion matrix)
import matplotlib.pyplot as plt
import seaborn as sns

X = df.drop(columns=["pitch_type_map"])
y = df["pitch_type_map"]

#resampling the data due to class imbalance

from imblearn.combine import SMOTETomek
smote_tomek = SMOTETomek(random_state=42)
X_resampled, y_resampled = smote_tomek.fit_resample(X, y)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.35, random_state=42)

class_mapping = {0: ' Fastball',
                1: 'Sinker',
                2: 'Cutter',
                3: 'Curveball',
                4: 'Slider',
                5: 'Changeup',
                6: 'Knuckle'}

label_encoder = LabelEncoder()

# Fit and transform y_train and y_test
y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

KeyboardInterrupt: 

In [24]:

from pybaseball import  playerid_lookup
from pybaseball import  statcast_pitcher
from pybaseball import  statcast
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = statcast_pitcher('2023-04-01', '2023-09-30', 519242)

pitch_dict = {
        'FF':0, 'FA':0, 'FT':0,
        'SI':1, 'FC': 1,
            'CU':2,'KC':2,'CS':2,'EP':2,
            'SL':3, 'ST': 3,
            'CH':4,'FS':4,'FO':4,'SC':4,
            'KN':5, 'GY':5, 
            
            'PO':np.nan}

# Map old pitch types to new mapping
df = df.sort_values(by=['game_pk', 'at_bat_number', 'pitch_number'])
df['pitch_id'] = df['game_pk'].astype(str) + "-" + df['at_bat_number'].astype(str) + "-" + df['pitch_number'].astype(str)
# df['pa_id'] = str(df['game_pk']) + "-" + str(df['at_bat_number'])
# df['pitch_id'] = str(df['pa_id']) + "-" + str(df['pitch_number'])
df['pitch_type_map'] = df['pitch_type'].map(pitch_dict)
fastball_maps = [0, 1]
df['is_fastball'] = np.where(df['pitch_type_map'].isin(fastball_maps), 1, 0)
#dropping rows with Nan

df = df.dropna(subset=['pitch_type_map'])
df['pitch_type_map'] = df['pitch_type_map'].astype(int)
df.dropna(subset=['pitch_type_map'], inplace = True)


#add lag pitches
df['prev_pitch_1'] = df['pitch_type_map'].shift(1)
df['prev_pitch_2'] = df['pitch_type_map'].shift(2)
df['prev_pitch_3'] = df['pitch_type_map'].shift(3)

df['on_1b'] = df['on_1b'] .fillna(0)
df['on_2b'] = df['on_2b'] .fillna(0)
df['on_3b'] = df['on_3b'] .fillna(0)

df['runners_on_base'] = df['on_1b'] + df['on_2b'] + df['on_3b']

df['runners_on_base'].value_counts()
df['runners_on_base'] = np.where(df['runners_on_base'] > 0.0001, 1, 0)



#current_pitch = row['pitch_id']
        
df["fb_prop_last_10"] = (
    df["is_fastball"]
    .shift(1)               
    .rolling(15, min_periods=1)
    .mean()
    )
   

Gathering Player Data


C:\Users\cstone\AppData\Local\Temp\ipykernel_30552\3483822403.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['pitch_id'] = df['game_pk'].astype(str) + "-" + df['at_bat_number'].astype(str) + "-" + df['pitch_number'].astype(str)
C:\Users\cstone\AppData\Local\Temp\ipykernel_30552\3483822403.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['pitch_type_map'] = df['pitch_type'].map(pitch_dict)
C:\Users\cstone\AppData\Local\Temp\ipykernel_30552\3483822403.py:28: PerformanceWarning: DataFrame is highly fragmented.  This

In [25]:
df

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches,pitch_id,pitch_type_map,is_fastball,prev_pitch_1,prev_pitch_2,prev_pitch_3,runners_on_base,fb_prop_last_10
68,FF,2023-09-28,91.1,3.24,5.14,"Sale, Chris",669720,519242,NaN,foul,...,46.198483,21.991037,716402-5-1,0,1,NaN,NaN,NaN,0,NaN
67,CH,2023-09-28,84.1,3.25,4.82,"Sale, Chris",669720,519242,field_out,hit_into_play,...,41.249104,27.899263,716402-5-2,4,0,0.0,NaN,NaN,0,1.000000
66,FF,2023-09-28,90.9,3.35,5.20,"Sale, Chris",668939,519242,NaN,ball,...,NaN,NaN,716402-6-1,0,1,4.0,0.0,NaN,0,0.500000
65,CH,2023-09-28,83.1,3.24,4.98,"Sale, Chris",668939,519242,NaN,ball,...,NaN,NaN,716402-6-2,4,0,0.0,4.0,0.0,0,0.666667
64,SL,2023-09-28,74.2,3.36,5.27,"Sale, Chris",668939,519242,NaN,called_strike,...,NaN,NaN,716402-6-3,3,0,4.0,0.0,4.0,0,0.500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1602,SL,2023-04-01,80.9,3.09,5.35,"Sale, Chris",663630,519242,hit_by_pitch,hit_by_pitch,...,NaN,NaN,718761-26-5,3,0,0.0,4.0,0.0,0,0.666667
1601,CH,2023-04-01,84.8,3.23,5.06,"Sale, Chris",602104,519242,NaN,foul,...,NaN,NaN,718761-27-1,4,0,3.0,0.0,4.0,1,0.600000
1600,FF,2023-04-01,92.8,3.24,5.24,"Sale, Chris",602104,519242,NaN,ball,...,NaN,NaN,718761-27-2,0,1,4.0,3.0,0.0,1,0.533333
1599,CH,2023-04-01,85.3,3.31,5.03,"Sale, Chris",602104,519242,NaN,swinging_strike,...,NaN,NaN,718761-27-3,4,0,0.0,4.0,3.0,1,0.533333
